In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv("Student Social Media And Mental Health Impact.csv") 
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
df.dropna(axis = 1)

In [ ]:
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

In [ ]:
df.columns.to_list()

# EDA


In [ ]:
df['Mental_Health_Score'].unique().mean()

In [ ]:
#Heat Map 
sns.heatmap(df.corr(numeric_only = True) , annot = True  , cmap = 'crest' , cbar = True)
plt.title("Heat MAp")
plt.show()

In [ ]:
#HistoPlot by Metal_Health_Score
plt.figure(figsize = (10 ,6))
sns.histplot(data = df , x = 'Mental_Health_Score' , kde = True  , palette = 'rocket')
plt.title("Mental Health Score ")
plt.xlabel("Scores")
plt.show()

In [ ]:
plt.figure(figsize=(14 , 25))
def plotting(var , num):
    plt.subplot(7 , 2 , num)
    sns.histplot(data = df, x = var , kde = True , bins = 10)

plotting('Age', 1)
plotting('Gender', 2)
plotting('Country', 3)
plotting('Academic_Level', 4)
plotting('Most_Used_Platform', 5)
plotting('Purpose_Of_Use', 6)
plotting('Avg_Daily_Usage_Hours', 7)
plotting('Daily_Unlocks', 8)
plotting('Study_Hours', 9)
plotting('Physical_Activity_Hours', 10)
plotting('Sleep_Hours_Per_Night',11)
plotting('Stress_Level', 12)
plotting('Mental_Health_Score' , 13)

plt.tight_layout()
plt.show()

In [ ]:
df['Country'].value_counts().index[:10].to_list()

In [ ]:
df['Stress_Level'].value_counts()

In [ ]:
#mental heath vs stress level
plt.figure(figsize=(10 , 6))
sns.boxplot(data = df , x = 'Stress_Level' , y = 'Mental_Health_Score', fill=False, gap=.1 , order = ['Low' , 'Medium' , 'High','Very High'])
plt.title('Stress Level vs Mental Health Score')
plt.xlabel('Stress Level')
plt.ylabel('Mental Health Score')

plt.show()

In [ ]:
df['Sleep_Hours_Per_Night'].value_counts().index[:10]

In [ ]:
#mentel HEalth vs  Sleep_Hours_Per_Night
plt.figure(figsize = (10 , 6))
sns.scatterplot(data = df , x = 'Sleep_Hours_Per_Night' , y = 'Mental_Health_Score')
plt.title('Mental vs Sleep Hours')
plt.xlabel('Sleep Hours')
plt.ylabel('Mental Health')

plt.show()

In [ ]:
df['Daily_Unlocks'].value_counts().unique()

In [ ]:
#mentel HEalth vs  Sleep_Hours_Per_Night
plt.figure(figsize = (10 , 6))
sns.scatterplot(data = df , x = 'Daily_Unlocks' , y = 'Mental_Health_Score')
plt.title('Daily_Unlocks vs Sleep Hours')
plt.xlabel('Daily_Unlocks')
plt.ylabel('Mental Health')

plt.show()

In [ ]:
df['Most_Used_Platform'].value_counts()

In [ ]:
#Most platform vs  Sleep_Hours_Per_Night
plt.figure(figsize = (10 , 6))
sns.barplot(data = df , x = 'Most_Used_Platform' , y = 'Mental_Health_Score' , hue = 'Gender')
plt.title('Most_Used_Platform vs Sleep Hours')
plt.xlabel('Most_Used_Platforms')
plt.ylabel('Mental Health')
plt.xticks(rotation=45)

plt.show()

In [ ]:
df['Physical_Activity_Hours'].value_counts().unique()

In [ ]:
plt.figure(figsize = (10 , 6))
sns.scatterplot(data = df , x = 'Physical_Activity_Hours' , y = 'Mental_Health_Score')
plt.title('Physical_Activity_Hours vs Sleep Hours')
plt.xlabel('Physical_Activity_Hours')
plt.ylabel('Mental Health')
plt.xticks(rotation=45)

plt.show()

## DATA CLEANING

In [ ]:
df.columns.to_list()

In [ ]:
df.describe()

In [ ]:
df = df.drop_duplicates()

In [ ]:
# checking outliers 
num_features = df.select_dtypes(include='number') #int64 or float64
Q1  = num_features.quantile(0.25)
Q3  = num_features.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = (num_features < lower_bound) | (num_features > upper_bound)
print(outliers.sum())

In [ ]:
#2. in Physical_Activity_Hours are Converting negative or unrealitstic value to realistic value
df['Physical_Activity_Hours'] = df['Physical_Activity_Hours'].clip(lower=0)

In [ ]:
df.describe()

##  SWekNess

In [ ]:
#skewNess is handle the 
# near to 0 -> ex: Centralized (0.01, 0.002)
# negative -> ex: Left Skewed (-1.56)
# poistive (greater than 0) -> ex: Right Skewed (1.256)

num_col = df.select_dtypes(include = 'number')
num_col.skew()

In [ ]:
# #Skewness Analysis: The numerical features show low levels of skewness.
# Most variables are approximately symmetric,
# with Study_Hours showing the highest positive skewness (0.436).
# Mental_Health_Score has a slight positive skewness of 0.207,
# indicating that its distribution is approximately symmetric with a small right tail.
#  Therefore, no major skewness correction is required at this stage.


## Feature Engineering


In [ ]:
#in the Data of column ['Country'] are very very differnt contry are there,
# how to Fix , i wann goes to ctreate a groupby top 10 country anmd S.no 11 are other create a handle and rest of 1 top 10 conaty are goes to other.
top_country = df['Country'].value_counts().index[:10].to_list()

In [ ]:
def group_country(Country):
    if Country in top_country:
        return Country
    else:
        return 'Other'

In [ ]:
df['group_country'] = df['Country'].apply(group_country)

In [ ]:
df['group_country'].value_counts()

## Train - Test Split|

In [ ]:
df.columns.to_list()

In [ ]:
from sklearn.model_selection import train_test_split

skew_col = ['Study_Hours'] # i gona chose skew cuz this value is 0.42 and goes to right skew that wayyyyy

numerical_col = ["Age", "Avg_Daily_Usage_Hours", "Daily_Unlocks","Physical_Activity_Hours",
                "Sleep_Hours_Per_Night"]

ordinal_col = ['Stress_Level'] 

normal_col = ["Gender", "Academic_Level", "Most_Used_Platform", "Purpose_Of_Use", "group_country"] # int this col Endong atre not only scaling

feature_col = skew_col + numerical_col+  ordinal_col + normal_col
X = df[feature_col]
Y = df['Mental_Health_Score']

x_train , x_test , y_train , y_test = train_test_split(X , Y, test_size= 0.2 , random_state= 42 , shuffle = True)

##  Preprocessing using ColumnTransformer

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import  FunctionTransformer , OneHotEncoder , OrdinalEncoder , StandardScaler
from sklearn.compose import ColumnTransformer

skew_pipeline = Pipeline(steps=[
    ("log_transform" , FunctionTransformer(np.log1p)),
    ('scaler' , StandardScaler())
])

numerical_pipeline = Pipeline(steps = [
    ("scaler" , StandardScaler())
])

ordinal_pipeline = Pipeline(steps = [
    ('encode' , OrdinalEncoder(categories=[['Low' , 'Medium' , 'High' , 'Very High']]))
])

normal_pipeline = Pipeline(steps = [
    ('encode' , OneHotEncoder(handle_unknown = "ignore"))
])

preprocessor = ColumnTransformer(transformers=[ ## which Pipeline and which Columns ##
    ('Skew_Pipeline' , skew_pipeline , skew_col),
    ("Numrical" , numerical_pipeline , numerical_col),
    ("Ordinal" , ordinal_pipeline  , ordinal_col),
    ('Normal' , normal_pipeline , normal_col)
])

## Model testing

In [ ]:
#1. LinearRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error , mean_squared_error , r2_score

LR_pipeline = Pipeline(steps = [
    ('preprocessing' , preprocessor),
    ('model' , LinearRegression(
        fit_intercept=True
    ))
])

LR_pipeline.fit(x_train , y_train)
LR_pipeline_pred_test = LR_pipeline.predict(x_test)
LR_pipeline_pred_train = LR_pipeline.predict(x_train)

print('Linear Model Test Prediction : ', LR_pipeline_pred_test[:10].round(2))
print('Linear Model Train Prediction : ', LR_pipeline_pred_train[:10].round(2))

In [ ]:
# Test Prediction By LR
LR_mae_test = mean_absolute_error(y_test , LR_pipeline_pred_test)
LR_mse_test = mean_squared_error(y_test , LR_pipeline_pred_test)
LR_r2_test = r2_score(y_test , LR_pipeline_pred_test)
LR_rmse_test = np.sqrt(LR_mse_test)

#------------------------------------------------------------------------------------------

#Train Prediction By LR
LR_mae_train = mean_absolute_error(y_train , LR_pipeline_pred_train)
LR_mse_train = mean_squared_error(y_train , LR_pipeline_pred_train)
LR_r2_train = r2_score(y_train , LR_pipeline_pred_train)
LR_rmse_train = np.sqrt(LR_mse_train)

In [ ]:
print("========== Linear Regression ==========")

print("\nTrain Performance:")
print("MAE  :", LR_mae_train)
print("MSE  :", LR_mse_train)
print("RMSE :", LR_rmse_train)
print("R²   :", LR_r2_train)

print("\nTest Performance:")
print("MAE  :", LR_mae_test)
print("MSE  :", LR_mse_test)
print("RMSE :", LR_rmse_test)
print("R²   :", LR_r2_test)

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x = y_test,
    y = LR_pipeline_pred_test
)

plt.plot(
    [y_test.min() , y_test.max()],
    [y_test.min() , y_test.max()],
    linestyle = '--'
)

plt.xlabel('Actual Mental Health Score')
plt.ylabel('Predicted Mental Health Score')
plt.title('Actual vs Predicted Mental Health Score')

plt.show()

In [ ]:
#Another Model RandomForestRegressor
from sklearn.ensemble import RandomForestRegressor
RFR_pipeline = Pipeline(steps = [
    ('preprocessing' , preprocessor),
    ('model' , RandomForestRegressor(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42
    ))
])

RFR_pipeline.fit(x_train , y_train)
RFR_pipeline_pred_test = RFR_pipeline.predict(x_test)
RFR_pipeline_pred_train = RFR_pipeline.predict(x_train)

print('Random Model Test Prediction : ', RFR_pipeline_pred_test[:10].round(2))
print('Random Model Train Prediction : ', RFR_pipeline_pred_train[:10].round(2))

In [ ]:
# Test Prediction By LR
RFR_mae_test = mean_absolute_error(y_test , RFR_pipeline_pred_test)
RFR_mse_test = mean_squared_error(y_test , RFR_pipeline_pred_test)
RFR_r2_test = r2_score(y_test , RFR_pipeline_pred_test)
RFR_rmse_test = np.sqrt(RFR_mse_test)

#------------------------------------------------------------------------------------------

#Train Prediction By LR
RFR_mae_train = mean_absolute_error(y_train , RFR_pipeline_pred_train)
RFR_mse_train = mean_squared_error(y_train , RFR_pipeline_pred_train)
RFR_r2_train = r2_score(y_train , RFR_pipeline_pred_train)
RFR_rmse_train = np.sqrt(RFR_mse_train)

In [ ]:
print("========== Random Forest Regression ==========")

print("\nTrain Performance:")
print("MAE  :", RFR_mae_train)
print("MSE  :", RFR_mse_train)
print("RMSE :", RFR_rmse_train)
print("R²   :", RFR_r2_train)

print("\nTest Performance:")
print("MAE  :", RFR_mae_test)
print("MSE  :", RFR_mse_test)
print("RMSE :", RFR_rmse_test)
print("R²   :", RFR_r2_test)

In [ ]:
plt.figure(figsize = (10 ,6))

sns.scatterplot(
    x = y_test,
    y = RFR_pipeline_pred_test
)
plt.plot(
    [y_test.min() , y_test.max()],
    [y_test.min() , y_test.max()],
    linestyle = '--'
)

plt.xlabel('Actual Mental Health Score')
plt.ylabel('Predicted Mental Health Score')
plt.title('Actual vs Predicted Mental Health Score')

plt.show()

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [5, 10, 15],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    estimator=RFR_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

In [ ]:
# fitting the all parameter and tuning Model
grid_search.fit(x_train , y_train)

In [ ]:
result = pd.DataFrame(grid_search.cv_results_)

best_results = result[
    [
        "param_model__max_depth",
        "param_model__min_samples_leaf",
        "param_model__min_samples_split",
        "param_model__n_estimators",
        "mean_test_score",
        "rank_test_score"
    ]
].sort_values("rank_test_score").head(10)

display(best_results)

In [ ]:
print("Best Params : " , grid_search.best_params_)
print("Best R2 Score : " , grid_search.best_score_ )


In [ ]:
Grid_RFR = grid_search.best_estimator_

Grid_RFR_pred_test = Grid_RFR.predict(x_test)
Grid_RFR_pred_train = Grid_RFR.predict(x_train)

print("Prediction Value of Grid Search on Test:",Grid_RFR_pred_test[:10].round(2))
print("Prediction Value of Grid Search on Train:",Grid_RFR_pred_train[:10].round(2))

In [ ]:
# Test Prediction By LR
Grid_RFR_mae_test = mean_absolute_error(y_test , Grid_RFR_pred_test)
Grid_RFR_mse_test = mean_squared_error(y_test , Grid_RFR_pred_test)
Grid_RFR_r2_test = r2_score(y_test , Grid_RFR_pred_test)
Grid_RFR_rmse_test = np.sqrt(Grid_RFR_mae_test)

#------------------------------------------------------------------------------------------

#Train Prediction By LR
Grid_RFR_mae_train = mean_absolute_error(y_train , Grid_RFR_pred_train)
Grid_RFR_mse_train = mean_squared_error(y_train , Grid_RFR_pred_train)
Grid_RFR_r2_train = r2_score(y_train , Grid_RFR_pred_train)
Grid_RFR_rmse_train = np.sqrt(RFR_mae_train)

In [ ]:
print("========== Random Forest Regression BY Grid Search CV ==========")

print("\nTest Performance:")
print("MAE  :", Grid_RFR_mae_test)
print("MSE  :", Grid_RFR_mse_test)
print("RMSE :", Grid_RFR_rmse_test)
print("R²   :", Grid_RFR_r2_test)

print("\nTrain Performance:")
print("MAE  :", Grid_RFR_mae_train)
print("MSE  :", Grid_RFR_mse_train)
print("RMSE :", Grid_RFR_rmse_train)
print("R²   :", Grid_RFR_r2_train)

In [ ]:
plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=y_test,
    y=Grid_RFR_pred_test
)

plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    linestyle='--'
)

plt.xlabel('Actual Mental Health Score')
plt.ylabel('Predicted Mental Health Score')
plt.title('Random Forest: Actual vs Predicted')

plt.show()

In [ ]:
residuals = y_test - Grid_RFR_pred_test

plt.figure(figsize=(8, 6))

sns.scatterplot(
    x=Grid_RFR_pred_test,
    y=residuals
)

plt.axhline(0, linestyle='--')

plt.xlabel('Predicted Mental Health Score')
plt.ylabel('Residual')
plt.title('Random Forest Residual Plot')

plt.show()

In [ ]:
#Checking Leagage in Pieline and also Dupliactes
print("Number of duplicate rows:", df.duplicated().sum())
print("Train shape:", x_train.shape)
print("Test shape :", x_test.shape)

In [ ]:
model_results = pd.DataFrame([
    
    # Linear Regression
    {
        'Model': 'Linear Regression',
        'Data': 'Train',
        'MAE': LR_mae_train,
        'MSE': LR_mse_train,
        'RMSE': LR_rmse_train,
        'R2': LR_r2_train
    },
    {
        'Model': 'Linear Regression',
        'Data': 'Test',
        'MAE': LR_mae_test,
        'MSE': LR_mse_test,
        'RMSE': LR_rmse_test,
        'R2': LR_r2_test
    },

    # Random Forest
    {
        'Model': 'Random Forest',
        'Data': 'Train',
        'MAE': RFR_mae_train,
        'MSE': RFR_mse_train,
        'RMSE': RFR_rmse_train,
        'R2': RFR_r2_train
    },
    {
        'Model': 'Random Forest',
        'Data': 'Test',
        'MAE': RFR_mae_test,
        'MSE': RFR_mse_test,
        'RMSE': RFR_rmse_test,
        'R2': RFR_r2_test
    },

    # GridSearch Random Forest
    {
        'Model': 'Tuned Random Forest',
        'Data': 'Train',
        'MAE': Grid_RFR_mae_train,
        'MSE': Grid_RFR_mse_train,
        'RMSE': Grid_RFR_rmse_train,
        'R2': Grid_RFR_r2_train
    },
    {
        'Model': 'Tuned Random Forest',
        'Data': 'Test',
        'MAE': Grid_RFR_mae_test,
        'MSE': Grid_RFR_mse_test,
        'RMSE': Grid_RFR_rmse_test,
        'R2': Grid_RFR_r2_test
    }
])

print(model_results)

In [ ]:
comparison = model_results.pivot(
    index='Model',
    columns='Data',
    values=['MAE', 'MSE', 'RMSE', 'R2']
)

comparison.round(4)


In [ ]:
# # Three regression models were evaluated for predicting Mental Health Score:
# Linear Regression, Random Forest Regression, and GridSearchCV-tuned Random Forest Regression.
# Among the evaluated models, Random Forest Regression achieved the best test performance 
# with an MAE of 0.3267, RMSE of 0.4428, and R² of 0.8902. Therefore, Random Forest Regression was selected as the final model for deployment.

## SAVE THE MODEL

In [ ]:
import joblib

joblib.dump(RFR_pipeline , 'Mental_Health.pkl')

In [ ]:
predictions = RFR_pipeline.predict(x_test)

print("Maximum prediction:", predictions.max())
print("Minimum prediction:", predictions.min())